In [ ]:
# 1. Install / Upgrade Dependencies (Run once per Colab session)
!pip install -q -U transformers datasets accelerate scikit-learn

import os
import time
import numpy as np
import pandas as pd
import torch
from google.colab import drive
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

In [ ]:
# 2. Mount Google Drive & Locate valid.csv
drive.mount('/content/drive')

# UPDATE THIS PATH to match where valid.csv lives in your Google Drive:
CSV_PATH = "/content/drive/MyDrive/valid.csv"

assert os.path.exists(CSV_PATH), f"File not found at {CSV_PATH}. Check your Drive folder path!"


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# 3. Configuration
MODEL_NAME = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"  # PubMedBERT
CONTEXT_WINDOW_WORDS = 60
MAX_LENGTH = 256
BATCH_SIZE = 16
EPOCHS = 12
LEARNING_RATE = 5e-5
SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)

In [ ]:
# 4. Load valid.csv & Create In-Memory Ambiguous Seen + Unseen Splits
raw_df = pd.read_csv(CSV_PATH, nrows=300000).dropna(subset=["TEXT", "LOCATION", "LABEL"])
raw_df["LOCATION"] = raw_df["LOCATION"].astype(int)

def get_acronym(row):
    words = str(row["TEXT"]).split()
    loc = int(row["LOCATION"])
    return words[loc] if 0 <= loc < len(words) else ""

raw_df["ACRONYM"] = raw_df.apply(get_acronym, axis=1)
raw_df = raw_df[raw_df["ACRONYM"] != ""]

# Keep only truly ambiguous acronyms (>= 2 competing expansions)
ambig_counts = raw_df.groupby("ACRONYM")["LABEL"].nunique()
ambiguous_acronyms = ambig_counts[ambig_counts >= 2].index
ambig_df = raw_df[raw_df["ACRONYM"].isin(ambiguous_acronyms)].copy()

# Keep labels with >= 30 instances so every sense has plenty of train & test examples
label_counts = ambig_df["LABEL"].value_counts()
valid_labels = label_counts[label_counts >= 30].index
ambig_df = ambig_df[ambig_df["LABEL"].isin(valid_labels)].copy()

# Rank ambiguous acronyms by total frequency and take the top 40 most common ambiguous acronyms
top_ambig_acronyms = (
    ambig_df.groupby("ACRONYM")
    .filter(lambda g: g["LABEL"].nunique() >= 2)["ACRONYM"]
    .value_counts()
    .head(40)
    .index.tolist()
)

# Split those 40 ambiguous acronyms into 30 Seen (75%) and 10 Unseen (25%)
np.random.seed(SEED)
np.random.shuffle(top_ambig_acronyms)
seen_acronyms = set(top_ambig_acronyms[:30])
unseen_acronyms = set(top_ambig_acronyms[30:])

seen_pool = ambig_df[ambig_df["ACRONYM"].isin(seen_acronyms)].copy()
unseen_pool = ambig_df[ambig_df["ACRONYM"].isin(unseen_acronyms)].copy()

# Create 1,000 Train / 500 Seen Test / 500 Unseen Test purely in RAM
sampled_seen, _ = train_test_split(
    seen_pool,
    train_size=1500,
    stratify=seen_pool["LABEL"],
    random_state=SEED,
)
train_df, test_seen_df = train_test_split(
    sampled_seen,
    train_size=1000,
    test_size=500,
    stratify=sampled_seen["LABEL"],
    random_state=SEED,
)
test_unseen_df = unseen_pool.sample(n=min(500, len(unseen_pool)), random_state=SEED).copy()

print(f"Seen Acronyms: {len(seen_acronyms)} ({train_df['LABEL'].nunique()} classes) | Unseen Acronyms: {len(unseen_acronyms)}")
print(f"In-Memory Splits -> Train: {len(train_df)} | Seen Test: {len(test_seen_df)} | Unseen Test: {len(test_unseen_df)}")

# Closed-vocabulary label mapping built strictly from train_df
unique_labels = sorted(train_df["LABEL"].unique())
label2id = {label: idx for idx, label in enumerate(unique_labels)}
id2label = {idx: label for label, idx in label2id.items()}


Seen Acronyms: 30 (204 classes) | Unseen Acronyms: 10
In-Memory Splits -> Train: 1000 | Seen Test: 500 | Unseen Test: 500


In [ ]:
# 5. Format Context Windows Around LOCATION
def format_medal_context(row, window=CONTEXT_WINDOW_WORDS):
    words = str(row["TEXT"]).split()
    loc = int(row["LOCATION"])
    if loc < 0 or loc >= len(words):
        return str(row["TEXT"])
    words[loc] = f"[E] {words[loc]} [/E]"
    start = max(0, loc - window)
    end = min(len(words), loc + window + 1)
    return " ".join(words[start:end])

for df_split in (train_df, test_seen_df, test_unseen_df):
    df_split["marked_text"] = df_split.apply(format_medal_context, axis=1)

train_df["label_id"] = train_df["LABEL"].map(label2id)
test_seen_df["label_id"] = test_seen_df["LABEL"].map(label2id)
test_unseen_df["label_id"] = test_unseen_df["LABEL"].map(label2id).fillna(-100).astype(int)


In [ ]:
# 6. Tokenize & Initialize Model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.add_special_tokens({"additional_special_tokens": ["[E]", "[/E]"]})

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(unique_labels),
    id2label=id2label,
    label2id=label2id,
)
model.resize_token_embeddings(len(tokenizer))

def tokenize_fn(batch):
    tokenized = tokenizer(batch["marked_text"], truncation=True, max_length=MAX_LENGTH)
    tokenized["labels"] = batch["label_id"]
    return tokenized

train_ds = Dataset.from_pandas(train_df[["marked_text", "label_id"]], preserve_index=False).map(tokenize_fn, batched=True)
test_seen_ds = Dataset.from_pandas(test_seen_df[["marked_text", "label_id"]], preserve_index=False).map(tokenize_fn, batched=True)
test_unseen_ds = Dataset.from_pandas(test_unseen_df[["marked_text", "label_id"]], preserve_index=False).map(tokenize_fn, batched=True)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect ide

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [ ]:
# 7. Train & Evaluate (All temporary files stay in Colab /content/, zero Drive writes)
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "exact_match_accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro", zero_division=0),
        "weighted_f1": f1_score(labels, preds, average="weighted", zero_division=0),
    }

training_args = TrainingArguments(
    output_dir="/content/tmp_bert_baseline",
    eval_strategy="epoch",
    save_strategy="no",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    report_to="none",
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_seen_ds,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

start_train = time.time()
trainer.train()
train_time = time.time() - start_train

# Evaluate on Seen Ambiguous Test Split
start_eval = time.time()
seen_metrics = trainer.evaluate()
eval_time = time.time() - start_eval

# Evaluate on Unseen Zero-Shot Test Split
unseen_out = trainer.predict(test_unseen_ds)
unseen_preds = np.argmax(unseen_out.predictions, axis=-1)
pred_strings = [id2label[p] for p in unseen_preds]
unseen_exact_match = np.mean([p == t for p, t in zip(pred_strings, test_unseen_df["LABEL"])])

peak_vram_mb = torch.cuda.max_memory_allocated() / (1024 ** 2) if torch.cuda.is_available() else 0.0

print("\n" + "="*52)
print("PubMedBERT Baseline Results (Zero Drive Writes)")
print("="*52)
print(f"GPU Used:                  {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"Training Wall Time:        {train_time:.2f} sec")
print(f"Inference Latency:         {(eval_time/len(test_seen_df))*1000:.2f} ms/sample")
print(f"Peak GPU VRAM:             {peak_vram_mb:.1f} MB")
print("-" * 52)
print(f"Seen Ambiguous Accuracy:   {seen_metrics['eval_exact_match_accuracy']*100:.2f}%")
print(f"Seen Macro F1:             {seen_metrics['eval_macro_f1']*100:.2f}%")
print(f"Unseen Zero-Shot Accuracy: {unseen_exact_match*100:.2f}%")
print("="*52)


Epoch,Training Loss,Validation Loss,Exact Match Accuracy,Macro F1,Weighted F1
1,5.357715,5.269734,0.010000,0.003496,0.004279
2,5.187021,4.821801,0.176000,0.123381,0.129920
3,4.797852,4.409922,0.316000,0.251110,0.258112
4,3.928286,4.032351,0.394000,0.316937,0.326984
5,3.499934,3.751295,0.438000,0.362828,0.370913
6,3.199755,3.499908,0.472000,0.399699,0.409269
7,2.898030,3.307411,0.488000,0.414769,0.425760
8,2.411268,3.154263,0.514000,0.441192,0.454554
9,2.239261,3.025496,0.528000,0.463080,0.472845
10,2.075053,2.949828,0.534000,0.466904,0.476519


Training Loss,Validation Loss,Epoch,Exact Match Accuracy,Macro F1,Weighted F1
1.838792,2.879092,12,0.544000,0.484144,0.491651



PubMedBERT Baseline Results (Zero Drive Writes)
GPU Used:                  Tesla T4
Training Wall Time:        154.84 sec
Inference Latency:         2.39 ms/sample
Peak GPU VRAM:             2808.3 MB
----------------------------------------------------
Seen Ambiguous Accuracy:   54.40%
Seen Macro F1:             48.41%
Unseen Zero-Shot Accuracy: 0.00%
